In [1]:
# Import Libraries

import os
import glob
import random
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
import gradio as gr

C:\Users\Acer\AppData\Local\Temp\ipykernel_27060\3302843475.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
e:\Vehicle_RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

KNOWLEDGE_BASE = "vehicle_knowledge_base"
DB_NAME = "vector_db"

In [3]:
# STAGE 1

documents = []
folders = sorted(glob.glob(f"{KNOWLEDGE_BASE}/*"))

print("=" * 60)
print("Loading Knowledge Base...")
print("=" * 60)

for folder in folders:
    category = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )

    folder_docs = loader.load()

    print(f"\nCategory : {category}")
    print(f"Documents: {len(folder_docs)}")

    for doc in folder_docs:
        relative_source = os.path.relpath(
            doc.metadata["source"],
            KNOWLEDGE_BASE
        ).replace("\\", "/")

        filename = os.path.basename(relative_source)

        # -------------------------------------
        # Extract markdown title
        # -------------------------------------

        title = filename.replace(".md", "").replace("_", " ").title()
        lines = doc.page_content.splitlines()

        for line in lines:
            if line.strip().startswith("# "):
                title = line.replace("#", "").strip()
                break

        # -------------------------------------
        # Store metadata
        # -------------------------------------

        doc.metadata = {

            "category": category,
            "source": relative_source,
            "filename": filename,
            "title": title

        }

        # -------------------------------------
        # Inject metadata into text
        # -------------------------------------

        enhanced_text = f"""
Document Title: {title}
Category: {category}
Source File: {relative_source}
------------------------------------------------------------

{doc.page_content}
"""
        doc.page_content = enhanced_text.strip()
        documents.append(doc)

print("\n" + "=" * 60)
print(f"Total Documents Loaded : {len(documents)}")
print("=" * 60)

Loading Knowledge Base...

Category : company_information
Documents: 2

Category : fleet_catalog
Documents: 4

Category : rental_operations
Documents: 3

Category : sales_and_finance
Documents: 4

Category : service_and_warranty
Documents: 2

Total Documents Loaded : 15


In [4]:
# ==========================================
# STAGE 2 - DOCUMENT CHUNKING
# ==========================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

print("=" * 60)
print("Splitting Documents...")
print("=" * 60)

text_splitter = RecursiveCharacterTextSplitter(

    separators=[
        "\n# ", "\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""
    ],

    chunk_size=900,
    chunk_overlap=200,
    length_function=len,
    keep_separator=True

)

chunks = text_splitter.split_documents(documents)

print(f"Total Chunks Created : {len(chunks)}")

print("\nSample Chunk Metadata")

print(chunks[0].metadata)

print("\nFirst 400 Characters\n")

print(chunks[0].page_content[:400])

Splitting Documents...
Total Chunks Created : 59

Sample Chunk Metadata
{'category': 'company_information', 'source': 'company_information/contact_information.md', 'filename': 'contact_information.md', 'title': 'Contact Information'}

First 400 Characters

Document Title: Contact Information
Category: company_information
Source File: company_information/contact_information.md
------------------------------------------------------------


In [5]:
# STAGE 3

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

if os.path.exists(DB_NAME):
    print("Loading existing database...")
    
    vector_store = Chroma(
        persist_directory=DB_NAME,
        embedding_function=embeddings
    )

else:
    print("Creating vector database...")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME,
    )

print(f"Vectors in database: {vector_store._collection.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 844.84it/s]


Loading existing database...
Vectors in database: 59


In [6]:

SIMILARITY_THRESHOLD = 0.45

def retrieve_from_domain(query: str, category: str, k: int = 3) -> str:
    """
    Retrieves relevant information from a specific category.
    """

    try:

        results = vector_store.similarity_search_with_relevance_scores(
            query=query,
            k=k,
            filter={"category": category}
        )

    except Exception:
        results = []
    if len(results) == 0:
        return ""

    filtered_docs = []
    for doc, score in results:

        if score >= SIMILARITY_THRESHOLD:
            filtered_docs.append(doc)

    if len(filtered_docs) == 0:
        return ""

    context = []

    for doc in filtered_docs:

        context.append(
            f"""
SOURCE: {doc.metadata['source']}

TITLE: {doc.metadata['title']}

CONTENT:
{doc.page_content}
"""
        )

    return "\n\n".join(context)

In [7]:
# STAGE 5
@tool
def query_fleet_catalog(query: str) -> str:
    """
    Search the dealership fleet catalog.
    Use this tool whenever the user asks about ANY vehicle model.
    This includes:
      vehicle names, model information, vehicle overview, specifications, engine, transmission, drivetrain, hybrid vehicles, electric vehicles, SUVs, sedans,
      crossover vehicles, fuel economy, cargo capacity, passenger capacity, pricing, safety features, technology, performance
    Always use this tool if a vehicle model name appears.
    """
    return retrieve_from_domain(query, "fleet_catalog")

@tool
def query_sales_and_finance(query: str) -> str:
    """
    Search dealership sales and finance policies.
    Includes:
     Financing, APR, credit score, rebates, promotions, incentives, EV tax credits, trade-ins, lease offers, payment rules, loan requirements, monthly income requirements
    """
    return retrieve_from_domain(query, "sales_and_finance")

@tool
def query_rental_operations(query: str) -> str:
    """
    Search rental policies.
    Includes:
      rental prices, rental eligibility, age requirements, driver's license, insurance, deposits, additional drivers, rental duration, mileage limits
    """
    return retrieve_from_domain(query, "rental_operations")

@tool
def query_service_and_warranty(query: str) -> str:
    """
    Search service and warranty information.
    Includes:
      warranty, maintenance, oil changes, service schedule, tire rotation, repairs, roadside assistance, inspections
    """
    return retrieve_from_domain(query, "service_and_warranty")

@tool
def query_company_information(query: str) -> str:
    """
    Search dealership company information.

    Includes:
      showroom, dealership location, phone numbers, email, opening hours, business hours, contact information
    """
    return retrieve_from_domain(query, "company_information")

tools = [
    query_fleet_catalog,
    query_sales_and_finance,
    query_rental_operations,
    query_service_and_warranty,
    query_company_information
]

In [ ]:
# ==========================================
# STAGE 6 - LLM & AGENT PROMPT
# ==========================================
llm = ChatGroq(
    model= "openai/gpt-oss-20b", temperature=0)

SYSTEM_PROMPT = """
You are Velocity Dealership's official AI assistant.

Your ONLY responsibility is to answer questions using information retrieved from
the dealership knowledge base through the available tools.

--------------------------------------------------
ALLOWED TOPICS
--------------------------------------------------

You may answer questions related to:

Vehicle models, Vehicle specifications, Vehicle overview, Hybrid vehicles, Electric vehicles, Fuel economy, Engine, Pricing, 
Financing, Trade-ins, Rebates, Rental policies, Warranty, Service schedules, Company information, Showroom locations, Business hours

--------------------------------------------------
TOOL USAGE RULES
--------------------------------------------------

Before answering ANY dealership-related question:

1. ALWAYS use one or more tools.
2. NEVER answer dealership questions from your own knowledge.
3. If a vehicle model name appears, ALWAYS call query_fleet_catalog first.
4. If the user asks about multiple topics, call multiple tools before answering.

Example:

Question:
Tell me about the EcoPulse H2 Hybrid and its financing.

Correct behavior:

1. query_fleet_catalog
2. query_sales_and_finance

Then combine the information.

--------------------------------------------------
WHEN INFORMATION IS MISSING
--------------------------------------------------

If the tools return no relevant information, reply ONLY:
"I don't have that specific information in the dealership knowledge base."

Do not guess.
Do not invent facts.

--------------------------------------------------
OUT OF SCOPE QUESTIONS
--------------------------------------------------

You are NOT a general-purpose chatbot.

If the question is unrelated to the dealership,
reply ONLY:
"I'm designed to answer questions about Velocity Dealerships. I can't assist with unrelated topics."

Examples of unrelated topics:

Weather, Politics, News, Programming, Mathematics, History, Sports, Movies, Celebrities, Medical advice, Recipes, Travel

Do not answer these questions.
Do not explain them.
Do not give external advice.

--------------------------------------------------
ANSWER STYLE
--------------------------------------------------

Keep answers:

• Professional
• Helpful
• Concise
• Accurate

Always cite the source document at the end.

Example:

Source:
fleet_catalog/hybrid_models.md
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)


In [9]:
# ==========================================
# STAGE 7 - BUILD AGENT
# ==========================================
agent = create_tool_calling_agent(llm, tools, prompt,)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,

    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4,
    return_intermediate_steps=False,
)